# 6 · A person in the loop

A node that needs a person raises `Ask`; its return type says what the answer is and how to ask. Only the calls downstream of it wait, and the run ends `waiting` with every question in `run.waiting`. `run.answer(waiting, value)` gives an answer and `run.resume()` goes on.

In [ ]:
from typing import Annotated

from conductor import Ask, Graph, GraphRecord, NodeRegistry, Param, Run, Series
from conductor.widgets import Textarea

registry = NodeRegistry()


@registry.node("text-uppercase", title="Uppercase", description="Turns a text into capital letters.", category="text")
def uppercase(text: str) -> str:
    return text.upper()


@registry.node("text-split", title="Split", description="Splits a text at a separator.", category="text")
def split(text: str, separator: str = ",") -> Series[str]:
    return text.split(separator)


@registry.node("text-join", title="Join", description="Joins the pieces into one text.", category="text")
def join(parts: Series[str], separator: str = "") -> str:
    return separator.join(parts)


Decision = Annotated[str, Param(title="Decision", widget=Textarea())]


@registry.node("approve", title="Approve", description="Asks a person to approve the proposal", category="review", asks=True)
def approve(proposal: Annotated[str, Param(title="Proposal", widget=Textarea())] = "") -> Decision:
    """Asks a person to approve a proposal; the person's answer is the node's output."""
    raise Ask("Approve or rewrite the proposal.", default=proposal)

## One question

`shout` waits for `approve`; `other` is called in the same `run()`.

In [ ]:
graph = Graph.compile(GraphRecord.model_validate({
    "nodes": {
        "approve": {"node": "approve", "inputs": {"proposal": {"constant": "Ship it"}}},
        "shout": {"node": "text-uppercase", "inputs": {"text": {"sources": [{"node": "approve", "output": "result"}]}}},
        "other": {"node": "text-uppercase", "inputs": {"text": {"constant": "independent"}}},
    },
    "outputs": {"shout": {"node": "shout", "output": "result"}, "other": {"node": "other", "output": "result"}},
}), registry)

run = graph.run()

print(run.status)
for waiting in run.waiting:
    print(waiting.id, waiting.loc[1:], waiting.prompt)
    print(waiting.questions)
print("called in this run():", sorted(node_id for node_id, outputs in run.results.items() if outputs["result"] is not None))

Answer, and resume:

In [ ]:
from conductor.events import RunEnd

answered = run.answer(run.waiting[0], "Ship it on Monday")

events = [event async for event in answered.stream()]
started = [event.loc[-1] for event in events if event.type == "node_start"]
ended = events[-1]  # the last event is the run's end, which carries the run
assert isinstance(ended, RunEnd)

print(ended.run.status)
print(ended.run.results["shout"]["result"])
print("started again:", started)

## A question per item

A node that runs once per item asks once per item. Each waiting call has its own id: the node, its item index and a fingerprint of the question asked there (`approve@1~3fa2c0d1`). An answer is given by it; `run.answer({id: value, ...})` gives several at once, refusing every wrong one together. A call left out goes on waiting.

In [ ]:
graph = Graph.compile(GraphRecord.model_validate({
    "nodes": {
        "drafts": {"node": "text-split", "inputs": {"text": {"constant": "first draft,second draft"}}},
        "approve": {"node": "approve", "inputs": {"proposal": {"sources": [{"node": "drafts", "output": "result"}]}}},
        "joined": {"node": "text-join", "inputs": {
            "parts": {"sources": [{"node": "approve", "output": "result"}]},
            "separator": {"constant": " | "},
        }},
    },
    "outputs": {"joined": {"node": "joined", "output": "result"}},
}), registry)

run = graph.run()
print(run.status, [(waiting.loc[1:], waiting.questions[0].default) for waiting in run.waiting])

run = run.answer({waiting.id: f"{waiting.questions[0].default}, approved" for waiting in run.waiting})

done = run.resume()
print(done.status, done.results["joined"]["result"])

Between calls of `resume()` a run is JSON, so it can wait days in a host's table:

```python
saved = run.to_json()
run = Run.from_json(saved, graph)        # in another process, with the same graph compiled again
```

## How the return type asks

`-> bool` asks a yes or no; the reference (`python -m conductor.about`) says what every type asks. A frozen dataclass is a form, one question per field: a field's own default is proposed, and `default=` and `choices=` on `Ask` go by field name, as the answer does; how each field is asked (its widget, its title) is its `Param`. Each question is the field's record as `describe()` writes it, plus what the `Ask` set, so a host draws it as it draws an input.

In [ ]:
import datetime
from dataclasses import dataclass
from typing import Literal

from conductor import Choice


@dataclass(frozen=True)
class Review:
    approved: bool
    note: Annotated[str | None, Param(widget=Textarea(rows=3))]
    due: datetime.date = datetime.date(2026, 12, 1)


@registry.node("review", title="Review", description="Asks a person to review a proposal.", asks=True)
def review(proposal: str) -> Review:
    raise Ask(f"Review: {proposal}")


@registry.node("labels", title="Labels", description="Asks a person which labels a proposal gets.", asks=True)
def labels(proposal: str) -> Series[Literal["urgent", "legal", "money"]]:
    raise Ask("Which labels?", default=["urgent"])


@registry.node("owner", title="Owner", description="Asks a person who owns a proposal.", asks=True)
def owner(proposal: str) -> str:
    raise Ask("Who owns it?", choices=[Choice("ida", "Ida B."), Choice("bo", "Bo K.")])


graph = Graph.compile(GraphRecord.model_validate({
    "nodes": {
        "r": {"node": "review", "inputs": {"proposal": {"constant": "Ship it"}}},
        "l": {"node": "labels", "inputs": {"proposal": {"constant": "Ship it"}}},
        "o": {"node": "owner", "inputs": {"proposal": {"constant": "Ship it"}}},
    },
    "outputs": {
        "approved": {"node": "r", "output": "approved"},
        "note": {"node": "r", "output": "note"},
        "labels": {"node": "l", "output": "result"},
        "owner": {"node": "o", "output": "result"},
    },
}), registry)
run = graph.run()
for waiting in run.waiting:
    print(waiting.id, waiting.prompt, [(q.name, q.required, q.default, q.choices) for q in waiting.questions])

A form is answered by field name. A field with a default may be left out, and so may one whose type takes `None` and that has no choices; a pick of several is a list of the choices. Every wrong answer is refused at once:

In [ ]:
from conductor import InvalidCallError

try:
    run.answer({"r": {"note": "fine"}, "l": ["urgent", "urgent"], "o": "carl"})
except InvalidCallError as wrong:
    print([(problem.loc, problem.ctx.get("question"), problem.ctx["reason"]) for problem in wrong.problems])

done = run.answer({"r": {"approved": True}, "l": ["urgent", "money"], "o": "ida"}).resume()
print(done.status, done.outputs)